# Kestrel — Colab Development Session

Follow cells **in order**. Do not skip steps.
See CLAUDE.md §29 for full protocol.

## Live Session (cells 1–6)
## Backtest Session (cells A–F)

---
## LIVE SESSION

In [ ]:
# Cell 1 — Clone repo
# Replace <REPO_URL> with your private repo SSH/HTTPS URL
!git clone <REPO_URL> kestrel
%cd kestrel

In [ ]:
# Cell 2 — Write .env from Colab secrets
# Add all required secrets in Colab → Secrets (key icon) before running
from google.colab import userdata

env_content = f"""ENV=dev
BOT_ID={userdata.get('BOT_ID')}
EXCHANGE={userdata.get('EXCHANGE')}
API_KEY={userdata.get('API_KEY')}
API_SECRET={userdata.get('API_SECRET')}
TESTNET=true
DB_HOST={userdata.get('DB_HOST')}
DB_PORT={userdata.get('DB_PORT', '5432')}
DB_NAME={userdata.get('DB_NAME')}
DB_USER={userdata.get('DB_USER')}
DB_PASSWORD={userdata.get('DB_PASSWORD')}
PAIR={userdata.get('PAIR', 'BTCUSDT')}
TIMEFRAME_ENTRY=5m
TIMEFRAME_REGIME=15m
LEVERAGE=20
BUCKET_SIZE_USDT=10.0
MAX_ACTIVE_BUCKETS=1
TELEGRAM_TOKEN={userdata.get('TELEGRAM_TOKEN')}
TELEGRAM_CHAT_ID={userdata.get('TELEGRAM_CHAT_ID')}
LOG_LEVEL=DEBUG
"""

with open('.env', 'w') as f:
    f.write(env_content)

print('.env written')

In [ ]:
# Cell 3 — Install + validate (must print [GO])
# If [NO-GO] is printed, fix the listed issue before continuing
!bash scripts/install.sh

In [ ]:
# Cell 4 — Start daemon in background (dev / simulation mode)
import subprocess, time
proc = subprocess.Popen(
    ['venv/bin/python', '-m', 'src.engine.daemon'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(3)
print(f'Daemon pid={proc.pid}')

In [ ]:
# Cell 5 — Health check
!bash scripts/status.sh

In [ ]:
# Cell 6 — Follow live event stream (run → interrupt to stop)
!bash scripts/logs.sh --follow

---
## BACKTEST SESSION (cells A–F)

In [ ]:
# Cell A — Fetch historical OHLCV (90 days, no auth required)
import ccxt, time

PAIR = 'BTC/USDT'
TIMEFRAME = '5m'
DAYS = 90

exchange = ccxt.binance()
since = int((time.time() - DAYS * 86400) * 1000)

ohlcvs = []
while True:
    batch = exchange.fetch_ohlcv(PAIR, TIMEFRAME, since=since, limit=1000)
    if not batch:
        break
    ohlcvs.extend(batch)
    since = batch[-1][0] + 1
    if len(batch) < 1000:
        break

print(f'Fetched {len(ohlcvs)} candles from {PAIR} {TIMEFRAME}')

In [ ]:
# Cell B — Build Candle objects + compute indicators
import sys; sys.path.insert(0, '.')
from src.config import Candle, compute_candle_geometry
from src.signal.indicators import compute_all_indicators
from src.config import load_params
from collections import deque

params = load_params('params.json')
BOT_ID = 'backtest-local'
buffer = deque(maxlen=120)
candles = []

for row in ohlcvs:
    ts, o, h, l, c, v = row
    geom = compute_candle_geometry(o, h, l, c)
    raw = Candle(bot_id=BOT_ID, ts=ts, pair='BTCUSDT', timeframe='5m',
                 open=o, high=h, low=l, close=c, volume=v, **geom)
    buffer.append(raw)
    inds = compute_all_indicators(list(buffer), ema_fast=params.ema_fast, ema_slow=params.ema_slow)
    full = Candle(bot_id=BOT_ID, ts=ts, pair='BTCUSDT', timeframe='5m',
                  open=o, high=h, low=l, close=c, volume=v,
                  **geom, **inds)
    buffer[-1] = full
    candles.append(full)

print(f'Built {len(candles)} candles with indicators')

In [ ]:
# Cell C — Indicator unit tests (sanity check)
from src.signal.indicators import compute_ema, compute_rsi, compute_atr

closes = [c.close for c in candles[-50:]]
ema9  = compute_ema(closes, 9)
ema21 = compute_ema(closes, 21)
rsi   = compute_rsi(closes, 14)
atr   = compute_atr(candles[-20:], 14)

assert 0 < rsi < 100, f'RSI out of range: {rsi}'
assert atr > 0, f'ATR must be positive: {atr}'
print(f'EMA9={ema9:.2f}  EMA21={ema21:.2f}  RSI14={rsi:.1f}  ATR14={atr:.2f}  ✓')

In [ ]:
# Cell D — Walk-forward backtest (60% train / 40% test)
import os
from dotenv import load_dotenv
load_dotenv()
from src.config import AppConfig
from src.backtest.runner import walk_forward

cfg = AppConfig.from_mapping(os.environ)
results = walk_forward(candles, params, cfg)

print('IN-SAMPLE:')
for k, v in results['in_sample'].items():
    print(f'  {k}: {v}')
print('\nOUT-OF-SAMPLE:')
for k, v in results['out_sample'].items():
    print(f'  {k}: {v}')

In [ ]:
# Cell E — Key metrics summary
oos = results['out_sample']
print(f"Win rate:       {oos['win_rate']*100:.1f}%  (target >55%)")
print(f"Sharpe ratio:   {oos['sharpe_ratio']:.2f}")
print(f"Max drawdown:   {oos['max_drawdown_pct']:.1f}%")
print(f"Avg PnL/trade:  ${oos['avg_pnl_usdt']:.4f}  (must > 0.18% of $10 = $0.018)")
print(f"Total trades:   {oos['total_trades']}")
print(f"Close reasons:  {oos['close_reasons']}")

In [ ]:
# Cell F — Equity curve
import matplotlib.pyplot as plt
from src.backtest.runner import run_backtest

split = int(len(candles) * 0.6)
test_result = run_backtest(candles[split:], params, cfg)

plt.figure(figsize=(12, 5))
plt.plot(test_result['equity_curve'], linewidth=1.2)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title('Out-of-Sample Equity Curve (USDT)')
plt.xlabel('Candle index')
plt.ylabel('Cumulative PnL (USDT)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()